# Bloque 2: Herramientas (Tools) - Dándole "manos" al modelo

Un LLM por sí solo es una caja aislada que solo predice texto. Para que se convierta en un Agente, necesita poder interactuar con el mundo exterior: consultar bases de datos, llamar APIs o escribir archivos.

En LangChain, convertimos cualquier función de Python en una "Herramienta" utilizando el decorador `@tool`. 

**Regla de Oro:** El modelo NUNCA lee tu código. Solo tiene acceso a:
1. El nombre de la función.
2. Los tipos de datos de los argumentos (*type hints*).
3. El *docstring* (la descripción de lo que hace la herramienta y sus parámetros).

In [ ]:
# Importamos el decorador mágico de LangChain
from langchain_core.tools import tool
import sqlite3
import json
import os
from datetime import datetime

# Nota: Asegúrate de haber ejecutado el script 'setup_datos.py' en la raíz 
# de tu proyecto para generar la base de datos y la agenda de prueba.

## Ejemplo 1: Consulta a una API externa (Lectura simple)
Esta herramienta demuestra cómo un agente puede obtener datos en tiempo real que no estaban en su entrenamiento original. Nota lo específico que es el *docstring*.

In [ ]:
@tool
def obtener_precio_accion(ticker: str) -> str:
    """
    Obtiene el precio actual de una acción en la bolsa de valores.
    Útil para responder preguntas sobre el mercado financiero.
    Args:
        ticker (str): El símbolo de la acción (ej. 'AAPL', 'AMZN', 'NFLX', 'NVDA').
    """
    precios_mock = {
        "AAPL": 185.50, "AMZN": 170.20, "NFLX": 605.10, "NVDA": 850.30
    }
    ticker = ticker.upper()
    if ticker in precios_mock:
        return f"El precio actual de {ticker} es de ${precios_mock[ticker]} USD."
    return f"No se pudo obtener el precio para el ticker {ticker}."

# Podemos ver cómo LangChain traduce nuestra función a un esquema para el LLM:
print("Esquema generado para el LLM:")
esquema = obtener_precio_accion.args_schema.model_json_schema()
print(json.dumps(esquema, indent=2))

In [ ]:
print("🧪 PRUEBA 1: Ticker válido (Éxito esperado)")
# Simulamos el diccionario 'args' que el LLM generaría
resultado_exito = obtener_precio_accion.invoke({"ticker": "NVDA"})
print(f"Resultado: {resultado_exito}")

print("-" * 40)

print("🧪 PRUEBA 2: Ticker no registrado (Manejo de error)")
# Simulamos un ticker que no está en nuestro mock
resultado_fallo = obtener_precio_accion.invoke({"ticker": "TSLA"})
print(f"Resultado: {resultado_fallo}")

## Ejemplo 2: Conexión a Base de Datos Local (SFA)
Aquí el agente se conecta a un sistema de información corporativo. Esto demuestra cómo encapsular la lógica de conexión y consulta SQL para que el LLM solo se preocupe por pasar el argumento correcto (la categoría).

In [ ]:
@tool
def consultar_inventario(categoria: str) -> str:
    """
    Consulta la base de datos SQL local de la empresa para buscar productos.
    Útil para saber qué artículos están disponibles, su stock y precios.
    Args:
        categoria (str): La categoría del producto (ej. 'Electrónica', 'Oficina').
    """
    db_path = "../inventario.db" # Asegúrate de haber ejecutado setup_datos.py
    
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT nombre, precio, stock FROM productos WHERE categoria = ?", (categoria,))
        resultados = cursor.fetchall()
        conn.close()
        
        if not resultados:
            return f"No se encontraron productos en la categoría: {categoria}."
            
        respuesta = f"Productos en {categoria}:\n"
        for nombre, precio, stock in resultados:
            respuesta += f"- {nombre}: ${precio} (Stock: {stock})\n"
        return respuesta
    except sqlite3.Error as e:
        return f"Error SQL: {str(e)}"

# --- Vemos el esquema actualizado (Pydantic V2) ---
print("Esquema generado para el LLM:")
esquema_inv = consultar_inventario.args_schema.model_json_schema()
print(json.dumps(esquema_inv, indent=2))

In [ ]:
print("🧪 PRUEBA 1: Categoría existente (Éxito)")
# 'Electrónica' debería existir si corrieron el setup_datos.py
resultado_exito = consultar_inventario.invoke({"categoria": "Electrónica"})
print(resultado_exito)

print("-" * 40)

print("🧪 PRUEBA 2: Categoría inexistente (Manejo de resultados vacíos)")
# Simulamos que el LLM busca una categoría que no vendemos
resultado_vacio = consultar_inventario.invoke({"categoria": "Jardinería"})
print(resultado_vacio)

## Ejemplo 3: Lectura de Archivos Locales (JSON)
Las herramientas pueden interactuar con el sistema de archivos del servidor. Esta herramienta le permite al agente leer la agenda del usuario y filtrar los eventos de un día específico.

In [ ]:
@tool
def consultar_agenda_dia(fecha: str) -> str:
    """
    Consulta los eventos y reuniones programadas en la agenda para un día específico.
    Args:
        fecha (str): La fecha exacta a consultar estrictamente en formato 'YYYY-MM-DD'.
    """
    archivo_agenda = "../agenda.json"
    
    if not os.path.exists(archivo_agenda):
        return "La agenda está vacía o el archivo no existe."
        
    try:
        with open(archivo_agenda, "r") as f:
            eventos = json.load(f)
            
        eventos_del_dia = [e for e in eventos if e.get("fecha_hora", "").startswith(fecha)]
        
        if not eventos_del_dia:
            return f"No tienes ningún evento programado para el {fecha}."
            
        respuesta = f"Agenda para el {fecha}:\n"
        for evento in eventos_del_dia:
            hora = evento["fecha_hora"].split(" ")[1]
            respuesta += f"- [{hora}] {evento['titulo']}: {evento['descripcion']}\n"
            
        return respuesta
    except Exception as e:
        return f"Error al leer la agenda: {str(e)}"

print("Esquema generado para el LLM:")
esquema_agenda = consultar_agenda_dia.args_schema.model_json_schema()
print(json.dumps(esquema_agenda, indent=2))

In [ ]:
print("🧪 PRUEBA 1: Día sin eventos (o archivo inexistente)")
# Probamos con una fecha lejana donde seguro no hay nada
resultado_vacio = consultar_agenda_dia.invoke({"fecha": "2029-12-31"})
print(resultado_vacio)

print("-" * 40)

print("🧪 PRUEBA 2: Día con eventos (Éxito)")
# Nota para la clase: Debes ajustar la fecha a un dia extra del actual
resultado_exito = consultar_agenda_dia.invoke({"fecha": "2026-03-05"})
print(resultado_exito)

## 🎯 Retos Prácticos (Ejercicios)

Ahora es tu turno de construir herramientas. Completa la lógica de las siguientes funciones asegurándote de:
1. Mantener el decorador `@tool`.
2. Escribir un *docstring* claro que le explique al LLM cuándo usarla y qué formato deben tener los argumentos.

### Reto 1: Manejo de Ambigüedad (Búsqueda de Clientes SFA)
En el mundo real, los usuarios son vagos al hacer peticiones (ej. "Busca a Carlos"). Si buscas "Carlos" en tu base de datos, encontrarás múltiples registros. Si un agente usa el correo incorrecto, puede ser un desastre corporativo.

**Tu objetivo:**
Crea la herramienta `buscar_cliente(nombre: str)` con una consulta SQL usando `LIKE`. 
El verdadero reto está en la lógica de Python y el Docstring:
1. Si la consulta devuelve **0 resultados**: Devuelve un string avisando que no existe.
2. Si devuelve **1 resultado**: Devuelve los datos de contacto.
3. Si devuelve **más de 1 resultado**: Tu herramienta NO debe devolver los correos. Debe devolver un string (ej. *"Múltiples coincidencias encontradas: Carlos Ruiz, Carlos Slim. Pídele al usuario que especifique el apellido."*) para obligar al agente a detener su ejecución, responderle al humano y esperar una aclaración.

In [ ]:
@tool
def buscar_cliente(nombre: str) -> str:
    """
    [TU RETO: Escribe aquí el docstring. Asegúrate de indicarle al LLM que 
    si recibe un mensaje de múltiples coincidencias, debe preguntarle al usuario.]
    """
    # --- CÓDIGO SQL PROPORCIONADO (NO MODIFICAR) ---
    import sqlite3, os
    db_path = "../inventario.db"
    if not os.path.exists(db_path): return "Error: Base de datos no encontrada."
    
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT nombre, empresa, email, telefono FROM clientes WHERE nombre LIKE ?", (f"%{nombre}%",))
        resultados = cursor.fetchall()
        conn.close()
    except sqlite3.Error as e:
        return f"Error SQL: {e}"
    # -----------------------------------------------
    
    # --- TU RETO: LÓGICA DEL AGENTE AQUÍ ---
    # Evalúa len(resultados) y devuelve el string apropiado para cada uno de los 3 casos:
    
    pass

In [ ]:
print("🧪 PRUEBA 1: Cliente que no existe")
print(buscar_cliente.invoke({"nombre": "Zanahoria"}))
print("-" * 40)

print("🧪 PRUEBA 2: Cliente único (Éxito)")
print(buscar_cliente.invoke({"nombre": "Ana"}))
print("-" * 40)

print("🧪 PRUEBA 3: Múltiples clientes (Ambigüedad intencional)")
print(buscar_cliente.invoke({"nombre": "Carlos"}))

### Reto 2: Agendar Reunión (Escritura de Archivos)
Crea una herramienta que reciba 3 parámetros (`titulo`, `fecha_hora`, `descripcion`) y guarde un nuevo evento en `agenda.json`.
 
*Tip: Deberás leer el archivo primero (si existe), agregar el nuevo diccionario a la lista, y volver a guardar el archivo con `json.dump`.*

In [ ]:
@tool
def agendar_reunion(titulo: str, fecha_hora: str, descripcion: str) -> str:
    """
    [Escribe aquí tu docstring. ¡Sé muy estricto con el formato que esperas 
    para 'fecha_hora' (ej. YYYY-MM-DD HH:MM) para que el LLM no se equivoque!]
    """
    pass

In [ ]:
print("🧪 PRUEBA: Agendando una nueva reunión")
resultado = agendar_reunion.invoke({
    "titulo": "Revisión de Arquitectura LangGraph",
    "fecha_hora": "2026-03-13 16:00",
    "descripcion": "Revisar los grafos construidos por los alumnos del CIMAT."
})
print(resultado)

print("-" * 40)
print("🔍 Verificando el archivo agenda.json...")
import json
try:
    with open("../agenda.json", "r") as f:
        eventos = json.load(f)
        print(f"Total de eventos en la agenda: {len(eventos)}")
        print(f"Último evento guardado: {eventos[-1]['titulo']}")
except Exception as e:
    print(f"Error al leer el archivo: {e}")